In [2]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

In [3]:
FEATURE_COLS = ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"]
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=FEATURE_COLS)

le = LabelEncoder()
y = le.fit_transform(df["ground_truth"])
groups = df["subject"].values
X = df[FEATURE_COLS].values
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

def build_stacking_model():
    base_learners = [
        ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("xgb", XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale_pos_weight,
                               eval_metric="logloss", random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42)),
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
    meta_learner = LogisticRegression(max_iter=1000, class_weight="balanced")
    return StackingClassifier(estimators=base_learners, final_estimator=meta_learner,
                               cv=5, passthrough=True)

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

N_REPEATS = 10
fold_f1 = []
all_preds, all_true = [], []

for repeat in range(N_REPEATS):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[train_idx])
        Xte = scaler.transform(X[test_idx])
        model = build_stacking_model()
        model.fit(Xtr, y[train_idx])
        preds = model.predict(Xte)
        all_preds.extend(preds)
        all_true.extend(y[test_idx])
        fold_f1.append(f1_score(y[test_idx], preds, average="macro", zero_division=0))

print(f"Proposed Stacking Ensemble: mean macro-F1 = {np.mean(fold_f1):.4f} (SD={np.std(fold_f1):.4f}), n={len(fold_f1)}")

Proposed Stacking Ensemble: mean macro-F1 = 0.7493 (SD=0.1119), n=50


In [5]:
summary = pd.read_csv("outputs/model_comparison_results.csv")
summary = pd.concat([summary, pd.DataFrame([{
    "Model": "Proposed_Stacking_Ensemble",
    "Mean_Macro_F1": np.mean(fold_f1),
    "Std": np.std(fold_f1)
}])], ignore_index=True).sort_values("Mean_Macro_F1", ascending=False)

print(summary.to_string(index=False))
summary.to_csv("outputs/model_comparison_results.csv", index=False)

print(classification_report(all_true, all_preds, target_names=le.classes_))

                     Model  Mean_Macro_F1      Std
Proposed_Stacking_Ensemble       0.749279 0.111944
        LogisticRegression       0.743812 0.130790
              RandomForest       0.730598 0.091181
          GradientBoosting       0.723334 0.113563
                       MLP       0.683362 0.109444
                RuleEngine       0.667595 0.129009
                       SVM       0.652526 0.119232
              precision    recall  f1-score   support

      NORMAL       0.88      0.81      0.84     14980
    STRESSED       0.62      0.74      0.68      6410

    accuracy                           0.79     21390
   macro avg       0.75      0.77      0.76     21390
weighted avg       0.80      0.79      0.79     21390



In [6]:
from scipy import stats

rf_scores = pd.read_csv("outputs/per_fold_f1_scores.csv")["RandomForest"].values

stat, p = stats.wilcoxon(fold_f1, rf_scores)
print(f"Stacking Ensemble vs RandomForest: p={p:.4f}")

Stacking Ensemble vs RandomForest: p=0.0156


In [ ]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
scores_df["Proposed_Stacking_Ensemble"] = fold_f1
scores_df.to_csv("outputs/per_fold_f1_scores.csv", index=False)
print("Added. Columns now:", scores_df.columns.tolist())

In [ ]:
import pandas as pd

summary = pd.read_csv("outputs/model_comparison_results.csv")

# Keep the BEST (highest F1) row per model, not just "last" — order in the file isn't reliable
summary = summary.sort_values("Mean_Macro_F1", ascending=False).drop_duplicates(subset="Model", keep="first")
summary = summary.sort_values("Mean_Macro_F1", ascending=False)
summary.to_csv("outputs/model_comparison_results.csv", index=False)
print(summary.to_string(index=False))

In [ ]:
import pandas as pd

summary = pd.read_csv("outputs/model_comparison_results.csv")
print("Raw file contents, in order:")
print(summary.to_string(index=False))
print("\nAll rows for Proposed_Stacking_Ensemble specifically:")
print(summary[summary["Model"] == "Proposed_Stacking_Ensemble"])

In [7]:
summary = pd.read_csv("outputs/model_comparison_results.csv")
summary["Model"] = summary["Model"].str.strip()  # remove any accidental whitespace

best_idx = summary.groupby("Model")["Mean_Macro_F1"].idxmax()
summary_clean = summary.loc[best_idx].sort_values("Mean_Macro_F1", ascending=False)

summary_clean.to_csv("outputs/model_comparison_results.csv", index=False)
print(summary_clean.to_string(index=False))

                     Model  Mean_Macro_F1      Std
Proposed_Stacking_Ensemble       0.749279 0.111944
        LogisticRegression       0.743812 0.130790
              RandomForest       0.730598 0.091181
          GradientBoosting       0.723334 0.113563
                       MLP       0.683362 0.109444
                RuleEngine       0.667595 0.129009
                       SVM       0.652526 0.119232


In [8]:
from sklearn.metrics import accuracy_score
import numpy as np

# Overall accuracy across all pooled predictions from the repeated CV run
overall_accuracy = accuracy_score(all_true, all_preds)
print(f"Overall accuracy (pooled across all folds): {overall_accuracy*100:.2f}%")

# Per-fold accuracy, if you want mean +/- SD to match your F1 reporting style
# (requires re-running the CV loop while tracking accuracy per fold instead of pooling)

Overall accuracy (pooled across all folds): 78.86%


In [9]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
import numpy as np

N_REPEATS = 10
fold_accuracy = []

for repeat in range(N_REPEATS):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[train_idx])
        Xte = scaler.transform(X[test_idx])
        model = build_stacking_model()
        model.fit(Xtr, y[train_idx])
        preds = model.predict(Xte)
        fold_accuracy.append(accuracy_score(y[test_idx], preds))

print(f"Accuracy: mean = {np.mean(fold_accuracy)*100:.2f}%, SD = {np.std(fold_accuracy)*100:.2f}%")

# Save alongside your existing F1 result
import pandas as pd
summary = pd.read_csv("outputs/model_comparison_results.csv")
summary.loc[summary["Model"] == "Proposed_Stacking_Ensemble", "Accuracy_Mean"] = np.mean(fold_accuracy) * 100
summary.loc[summary["Model"] == "Proposed_Stacking_Ensemble", "Accuracy_SD"] = np.std(fold_accuracy) * 100
summary.to_csv("outputs/model_comparison_results.csv", index=False)
print(summary.to_string(index=False))

Accuracy: mean = 78.88%, SD = 9.20%
                     Model  Mean_Macro_F1      Std  Accuracy_Mean  Accuracy_SD
Proposed_Stacking_Ensemble       0.749279 0.111944       78.88241      9.20239
        LogisticRegression       0.743812 0.130790            NaN          NaN
              RandomForest       0.730598 0.091181            NaN          NaN
          GradientBoosting       0.723334 0.113563            NaN          NaN
                       MLP       0.683362 0.109444            NaN          NaN
                RuleEngine       0.667595 0.129009            NaN          NaN
                       SVM       0.652526 0.119232            NaN          NaN
